# FPL Point Prediction Model — Training & Evaluation

Trains a LightGBM model (one per position) to predict `total_points` per player per gameweek.

**Pipeline:**
1. Build features from 9 seasons of data 
2. Run temporal cross-validation (expanding window, 6 folds)
3. Train final model on all data except holdout (2024-25)
4. Save model to disk for use in the RL environment

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path('../data')
MODEL_DIR = Path('../models/point_predictor')

SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25',
]
HOLDOUT = '2024-25'

## 1. Build Features

In [2]:
from fpl_rl.prediction.id_resolver import IDResolver
from fpl_rl.prediction.feature_pipeline import FeaturePipeline

id_resolver = IDResolver(DATA_DIR)
pipeline = FeaturePipeline(DATA_DIR, id_resolver, SEASONS)

print('Building features across all seasons...')
df = pipeline.build()
print(f'\nResult: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Positions: {dict(df["position"].value_counts())}')

Building features across all seasons...


Seasons:   0%|          | 0/9 [00:00<?, ?season/s]

Understat players:   0%|          | 0/683 [00:00<?, ?player/s]

Understat players:   0%|          | 0/647 [00:00<?, ?player/s]

FBref parquet not found: ..\data\fbref\2016-17_passing.parquet


...

Understat players:   0%|          | 0/865 [00:00<?, ?player/s]

FBref parquet not found: ..\data\fbref\2022-23_passing.parquet


Understat players:   0%|          | 0/784 [00:00<?, ?player/s]

FBref parquet not found: ..\data\fbref\2023-24_passing.parquet



Result: 215,087 rows x 81 columns
Seasons: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Positions: {'MID': np.int64(74588), 'DEF': np.int64(61543), 'FWD': np.int64(23111), 'GK': np.int64(20082)}


In [3]:
# Quick data quality check
print('=== Data Quality ===')
print(f'Target (total_points) NaN: {df["target"].isna().sum()}')
print(f'Position NaN: {df["position"].isna().sum()}')
print(f'\nTarget distribution:')
print(df['target'].describe().round(2))

# Feature NaN rates
feature_cols = [c for c in df.columns if c not in {'code','element','season','GW','position','target','total_points'}]
nan_pct = (df[feature_cols].isna().mean() * 100).sort_values(ascending=False)
print(f'\nFeature NaN rates (top 15):')
print(nan_pct.head(15).round(1))

=== Data Quality ===
Target (total_points) NaN: 0
Position NaN: 35763

Target distribution:
count    215087.00
mean          1.31
std           2.53
min          -7.00
25%           0.00
50%           0.00
75%           2.00
max          31.00
Name: target, dtype: float64

Feature NaN rates (top 15):
prev_blocks_per90        73.0
prev_prog_dist_per90     69.9
prev_pass_cmp_pct        67.4
fpl_xgc_rolling_5        63.6
fpl_xg_rolling_5         63.6
fpl_xa_rolling_5         63.6
starts_rolling_5         63.6
fpl_xgi_rolling_5        63.6
prev_tkl_int_per90       55.3
prev_sot_per90           52.4
prev_gls_per90           52.4
prev_xg_per90            49.5
prev_key_passes_per90    49.5
prev_npxg_per90          49.5
prev_xa_per90            49.5
dtype: float64


In [4]:
# Drop rows without position or target
df = df.dropna(subset=['position', 'target'])
print(f'After dropping NaN target/position: {len(df):,} rows')

After dropping NaN target/position: 179,324 rows


## 2. Temporal Cross-Validation

In [5]:
from fpl_rl.prediction.evaluation import TemporalCV

cv = TemporalCV(min_train_seasons=2, holdout=HOLDOUT)
folds = cv.generate_folds(df)
print(f'{len(folds)} folds generated\n')

for i, (train, val, test) in enumerate(folds):
    train_seasons = sorted(train['season'].unique())
    test_season = test['season'].iloc[0]
    print(f'Fold {i+1}: train={train_seasons} ({len(train):,}), '
          f'val={len(val):,}, test={test_season} ({len(test):,})')

6 folds generated

Fold 1: train=['2016-17', '2017-18'] (17,889), val=2,292, test=2018-19 (14,098)
Fold 2: train=['2016-17', '2017-18', '2018-19'] (31,425), val=2,854, test=2019-20 (18,308)
Fold 3: train=['2016-17', '2017-18', '2018-19', '2019-20'] (48,356), val=4,231, test=2020-21 (22,889)
Fold 4: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21'] (70,261), val=5,215, test=2021-22 (23,230)
Fold 5: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22'] (93,020), val=5,686, test=2022-23 (24,957)
Fold 6: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23'] (117,689), val=5,974, test=2023-24 (28,742)


In [6]:
print('Running temporal CV with DEFAULT params...\n')
results_default = cv.evaluate(df, params={'n_estimators': 500, 'verbose': -1})

print(f'Overall MAE:  {results_default["mae"]:.3f}')
print(f'Overall RMSE: {results_default["rmse"]:.3f}')
print(f'\nPer-position MAE:')
for pos, mae in sorted(results_default['per_position_mae'].items()):
    print(f'  {pos}: {mae:.3f}')

print(f'\nPer-fold results:')
for fold in results_default['fold_results']:
    print(f'  Fold {fold["fold"]} ({fold["test_season"]}): '
          f'MAE={fold["mae"]:.3f}, RMSE={fold["rmse"]:.3f}, n={fold["n_test"]:,}')

Running temporal CV with DEFAULT params...

Overall MAE:  1.253
Overall RMSE: 2.191

Per-position MAE:
  DEF: 1.363
  FWD: 1.429
  GK: 0.906
  MID: 1.204

Per-fold results:
  Fold 1 (2018-19): MAE=1.649, RMSE=2.516, n=14,098
  Fold 2 (2019-20): MAE=1.477, RMSE=2.335, n=18,308
  Fold 3 (2020-21): MAE=1.276, RMSE=2.220, n=22,889
  Fold 4 (2021-22): MAE=1.268, RMSE=2.245, n=23,230
  Fold 5 (2022-23): MAE=1.117, RMSE=2.053, n=24,957
  Fold 6 (2023-24): MAE=1.006, RMSE=1.963, n=28,742


## 2b. Hyperparameter Tuning

Random search over LightGBM hyperparameters using the last 3 CV folds for speed.
This typically takes 10-20 minutes for 50 trials.

In [7]:
from fpl_rl.prediction.tuning import random_search
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

print('Running hyperparameter tuning (50 trials, last 3 folds)...\n')
tuning_results = random_search(df, n_trials=50, last_n_folds=3, seed=42)

print(f'\n{"="*60}')
print(f'Best MAE: {tuning_results["best_mae"]:.4f}')
print(f'Default MAE: {results_default["mae"]:.4f}')
print(f'Improvement: {results_default["mae"] - tuning_results["best_mae"]:.4f}')
print(f'\nBest params:')
for k, v in sorted(tuning_results['best_params'].items()):
    print(f'  {k}: {v}')

# Show top 5 trials
trials_sorted = sorted(tuning_results['all_trials'], key=lambda t: t['mae'])
print(f'\nTop 5 trials:')
for t in trials_sorted[:5]:
    print(f'  Trial {t["trial"]}: MAE={t["mae"]:.4f} — {t["params"]}')

Fold 1: train=['2016-17', '2017-18'] (17889 rows), val=2292 rows, test=2018-19 (14098 rows)


Running hyperparameter tuning (50 trials, last 3 folds)...



Fold 2: train=['2016-17', '2017-18', '2018-19'] (31425 rows), val=2854 rows, test=2019-20 (18308 rows)
Fold 3: train=['2016-17', '2017-18', '2018-19', '2019-20'] (48356 rows), val=4231 rows, test=2020-21 (22889 rows)
Fold 4: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21'] (70261 rows), val=5215 rows, test=2021-22 (23230 rows)
Fold 5: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22'] (93020 rows), val=5686 rows, test=2022-23 (24957 rows)
Fold 6: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23'] (117689 rows), val=5974 rows, test=2023-24 (28742 rows)
Tuning: 50 trials, 3 folds (of 6 total)
Training with 75 features: ['pts_rolling_3', 'pts_rolling_5', 'pts_rolling_10', 'mins_rolling_3', 'mins_rolling_5', 'mins_std_5', 'goals_rolling_3', 'goals_rolling_5', 'assists_rolling_3', 'assists_rolling_5']
  GK: 8087 samples, train MAE=0.393
  DEF: 24883 samples, train MAE=1.047
  MID: 27598 samples, train MAE=0.964
  FWD: 9693 


Best MAE: 1.1109
Default MAE: 1.2535
Improvement: 0.1426

Best params:
  feature_fraction: 0.6
  lambda_l1: 0.1
  lambda_l2: 0.1
  learning_rate: 0.01
  min_child_samples: 100
  n_estimators: 2000
  num_leaves: 31

Top 5 trials:
  Trial 6: MAE=1.1109 — {'num_leaves': 31, 'learning_rate': 0.01, 'min_child_samples': 100, 'feature_fraction': 0.6, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'n_estimators': 2000}
  Trial 9: MAE=1.1124 — {'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 50, 'feature_fraction': 0.6, 'lambda_l1': 0.0, 'lambda_l2': 0.0, 'n_estimators': 1000}
  Trial 27: MAE=1.1140 — {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100, 'feature_fraction': 0.6, 'lambda_l1': 1.0, 'lambda_l2': 1.0, 'n_estimators': 2000}
  Trial 47: MAE=1.1140 — {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100, 'feature_fraction': 0.6, 'lambda_l1': 1.0, 'lambda_l2': 1.0, 'n_estimators': 1000}
  Trial 16: MAE=1.1151 — {'num_leaves': 31, 'learning_rate': 0.03, 'm

In [8]:
# Validate tuned params on ALL 6 folds (not just the 3 used during search)
best_params = {**{'verbose': -1}, **tuning_results['best_params']}
print(f'Validating tuned params on all 6 folds...\n')
results_tuned = cv.evaluate(df, params=best_params)

print(f'{"Metric":<25} {"Default":>10} {"Tuned":>10} {"Delta":>10}')
print('-' * 55)
print(f'{"Overall MAE":<25} {results_default["mae"]:>10.4f} {results_tuned["mae"]:>10.4f} {results_tuned["mae"] - results_default["mae"]:>+10.4f}')
print(f'{"Overall RMSE":<25} {results_default["rmse"]:>10.4f} {results_tuned["rmse"]:>10.4f} {results_tuned["rmse"] - results_default["rmse"]:>+10.4f}')
for pos in ['GK', 'DEF', 'MID', 'FWD']:
    d = results_default['per_position_mae'].get(pos, float('nan'))
    t = results_tuned['per_position_mae'].get(pos, float('nan'))
    print(f'{pos + " MAE":<25} {d:>10.4f} {t:>10.4f} {t - d:>+10.4f}')

Fold 1: train=['2016-17', '2017-18'] (17889 rows), val=2292 rows, test=2018-19 (14098 rows)
Fold 2: train=['2016-17', '2017-18', '2018-19'] (31425 rows), val=2854 rows, test=2019-20 (18308 rows)


Validating tuned params on all 6 folds...



Fold 3: train=['2016-17', '2017-18', '2018-19', '2019-20'] (48356 rows), val=4231 rows, test=2020-21 (22889 rows)
Fold 4: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21'] (70261 rows), val=5215 rows, test=2021-22 (23230 rows)
Fold 5: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22'] (93020 rows), val=5686 rows, test=2022-23 (24957 rows)
Fold 6: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23'] (117689 rows), val=5974 rows, test=2023-24 (28742 rows)
Training with 75 features: ['pts_rolling_3', 'pts_rolling_5', 'pts_rolling_10', 'mins_rolling_3', 'mins_rolling_5', 'mins_std_5', 'goals_rolling_3', 'goals_rolling_5', 'assists_rolling_3', 'assists_rolling_5']
  GK: 2037 samples, train MAE=1.245
  DEF: 6434 samples, train MAE=1.542
  MID: 6614 samples, train MAE=1.320
  FWD: 2804 samples, train MAE=1.615
  Fold 1 (2018-19): MAE=1.592, RMSE=2.462
Training with 75 features: ['pts_rolling_3', 'pts_rolling_5', 'pts_rolling_10'

Metric                       Default      Tuned      Delta
-------------------------------------------------------
Overall MAE                   1.2535     1.2296    -0.0239
Overall RMSE                  2.1913     2.1605    -0.0308
GK MAE                        0.9063     0.8887    -0.0176
DEF MAE                       1.3628     1.3468    -0.0160
MID MAE                       1.2037     1.1753    -0.0285
FWD MAE                       1.4290     1.3937    -0.0353


## 3. Train Final Model

Train on all seasons except the holdout (2024-25), using the last 8 GWs of 2023-24 as validation for early stopping.
Uses tuned hyperparameters from step 2b.

In [9]:
from fpl_rl.prediction.model import PointPredictor

# Split train / val
train_seasons = [s for s in SEASONS if s != HOLDOUT]
train_full = df[df['season'].isin(train_seasons)]

last_season = train_seasons[-1]
last_data = train_full[train_full['season'] == last_season]
val_cutoff = last_data['GW'].max() - 8

val_mask = (train_full['season'] == last_season) & (train_full['GW'] > val_cutoff)
val_df = train_full[val_mask]
train_df = train_full[~val_mask]

print(f'Training: {len(train_df):,} rows ({sorted(train_df["season"].unique())})')
print(f'Validation: {len(val_df):,} rows (last 8 GWs of {last_season})')
print(f'Using tuned params: {best_params}\n')

predictor = PointPredictor(
    params=best_params,
    early_stopping_rounds=50,
)
train_results = predictor.train(train_df, val_df)

print(f'\nTraining MAE per position:')
for pos, mae in sorted(train_results.items()):
    print(f'  {pos}: {mae:.3f}')

Training with 75 features: ['pts_rolling_3', 'pts_rolling_5', 'pts_rolling_10', 'mins_rolling_3', 'mins_rolling_5', 'mins_std_5', 'goals_rolling_3', 'goals_rolling_5', 'assists_rolling_3', 'assists_rolling_5']


Training: 145,605 rows (['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24'])
Validation: 6,800 rows (last 8 GWs of 2023-24)
Using tuned params: {'verbose': -1, 'num_leaves': 31, 'learning_rate': 0.01, 'min_child_samples': 100, 'feature_fraction': 0.6, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'n_estimators': 2000}



  GK: 16482 samples, train MAE=0.785
  DEF: 50328 samples, train MAE=1.217
  MID: 59561 samples, train MAE=1.096
  FWD: 19234 samples, train MAE=1.226



Training MAE per position:
  DEF: 1.217
  FWD: 1.226
  GK: 0.785
  MID: 1.096


In [10]:
# Feature importance (top 20)
fi = predictor.feature_importance()
print('Top 20 features by importance (gain):\n')
print(fi.head(20).to_string(index=False))

Top 20 features by importance (gain):

             feature   importance
      mins_rolling_3 1.169575e+06
       ict_rolling_3 6.196800e+05
       pts_rolling_3 3.991242e+05
        xg_rolling_3 1.892010e+05
              is_dgw 1.722532e+05
               value 1.418714e+05
       selected_norm 1.410392e+05
      ict_rolling_10 1.287977e+05
       bps_rolling_5 9.496592e+04
                 fdr 7.697711e+04
      mins_rolling_5 7.456886e+04
      season_avg_pts 7.341030e+04
        xa_rolling_3 7.277998e+04
   xgchain_rolling_5 7.052710e+04
  xgchain_rolling_10 6.039357e+04
 opp_pts_conceded_r5 6.038925e+04
       ict_rolling_5 5.909957e+04
opp_defence_strength 5.723868e+04
       goals_vs_xg_5 5.049859e+04
     assists_vs_xa_5 5.036032e+04


## 4. Evaluate on Holdout (2024-25)

In [11]:
holdout_df = df[df['season'] == HOLDOUT]

if holdout_df.empty:
    print(f'No holdout data for {HOLDOUT}')
else:
    preds = predictor.predict(holdout_df)
    actuals = holdout_df['target'].values
    errors = np.abs(preds - actuals)

    print(f'Holdout ({HOLDOUT}): {len(holdout_df):,} rows')
    print(f'MAE:  {errors.mean():.3f}')
    print(f'RMSE: {np.sqrt(np.mean((preds - actuals)**2)):.3f}')

    print(f'\nPer-position holdout MAE:')
    for pos in ['GK', 'DEF', 'MID', 'FWD']:
        mask = holdout_df['position'].values == pos
        if mask.any():
            print(f'  {pos}: {errors[mask].mean():.3f} (n={mask.sum():,})')

Holdout (2024-25): 26,919 rows
MAE:  0.992
RMSE: 1.908

Per-position holdout MAE:
  GK: 0.750 (n=2,827)
  DEF: 0.999 (n=9,024)
  MID: 1.001 (n=12,084)
  FWD: 1.164 (n=2,984)


## 5. Save Model

In [12]:
predictor.save(MODEL_DIR)
print(f'Model saved to {MODEL_DIR.resolve()}')
print(f'Files: {[f.name for f in MODEL_DIR.iterdir()]}')

# Verify load roundtrip
loaded = PointPredictor.load(MODEL_DIR)
test_preds = loaded.predict(holdout_df.head(10))
print(f'\nLoad roundtrip OK — sample predictions: {test_preds.round(2)}')

Saved 4 models to ..\models\point_predictor


Model saved to C:\Users\alexa\OneDrive\Documents\FPL-RL\models\point_predictor
Files: ['DEF.lgb', 'feature_names.json', 'FWD.lgb', 'GK.lgb', 'metadata.json', 'MID.lgb']


Loaded 4 models from ..\models\point_predictor (75 features)



Load roundtrip OK — sample predictions: [0.75 0.07 0.06 0.04 0.03 0.12 0.12 0.04 0.05 0.03]


## 6. Quick Sanity Checks

Spot-check predictions for well-known players.

In [13]:
if not holdout_df.empty:
    holdout_with_preds = holdout_df.copy()
    holdout_with_preds['predicted'] = preds
    holdout_with_preds['error'] = errors

    # Per-GW average prediction vs actual
    gw_summary = holdout_with_preds.groupby('GW').agg(
        avg_actual=('target', 'mean'),
        avg_predicted=('predicted', 'mean'),
        mae=('error', 'mean'),
    ).round(3)
    print('Per-GW summary (holdout):')
    print(gw_summary.to_string())

    # Top predicted players in a sample GW
    sample_gw = holdout_with_preds['GW'].median()
    gw_data = holdout_with_preds[holdout_with_preds['GW'] == sample_gw]
    top = gw_data.nlargest(10, 'predicted')[['code', 'position', 'predicted', 'target']]
    top['name'] = top['code'].apply(id_resolver.player_name)
    print(f'\nTop 10 predicted in GW {int(sample_gw)}:')
    print(top[['name', 'position', 'predicted', 'target']].to_string(index=False))

Per-GW summary (holdout):
    avg_actual  avg_predicted    mae
GW                                  
1        1.317          1.365  1.240
2        1.367          1.260  0.980
3        1.140          1.266  0.865
4        1.235          1.251  1.048
5        1.212          1.236  0.987
6        1.142          1.232  0.992
7        1.311          1.232  1.026
8        1.165          1.197  1.005
9        1.151          1.202  0.992
10       1.150          1.191  0.910
11       1.201          1.192  1.002
12       1.228          1.187  1.122
13       1.200          1.169  1.050
14       1.248          1.163  1.121
15       1.127          1.157  0.993
16       1.168          1.191  0.996
17       1.312          1.154  1.160
18       1.235          1.187  1.005
19       1.176          1.174  1.013
20       1.080          1.191  0.943
21       1.122          1.150  0.923
22       1.167          1.135  1.064
23       1.102          1.152  0.970
24       1.256          1.215  1.067
25       1.2

---

## Usage in the RL Environment

```python
from fpl_rl.env.fpl_env import FPLEnv

env = FPLEnv(
    season='2024-25',
    predictor_model_dir='models/point_predictor',
)
# features[18] in observations now contains predicted points / 15.0
```